# Preprocessing dan Data Cleaning

Notebook ini menggunakan implementasi aktual pada `src/data/clean.py`. Sumber data hanya `synthetic_generator/output/data_bank`; folder `kunci_evaluasi` tidak dibaca untuk proses cleaning.

Aturan utama: usia tidak diimputasi, kategori teks distandardisasi, duplikasi dihapus berdasarkan business key, outlier nominal x1000 dikoreksi, `jumlah_transaksi` memakai median pasangan, dan saldo akhir missing diisi dengan `saldo_rata2_rp * median_ratio`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import path, settings
from src.data.clean import clean_all, evaluate_cleaning, validate_cleaned

RAW = path('raw')
CLEAN = path('processed') / 'clean'
print('Project root:', ROOT)
print('Raw data   :', RAW)
print('Clean data :', CLEAN)

## 1. Audit data mentah

Missing tidak langsung dianggap sebagai error. Missing pada `bulan_keluar` atau atribut internal perusahaan non-nasabah dapat bersifat struktural, sedangkan missing pada nominal/produk perlu ditangani sesuai aturan cleaning.

In [ ]:
FILES = sorted(p for p in RAW.iterdir() if p.suffix in {'.csv', '.parquet'})
def read_raw(file):
    return pd.read_parquet(file) if file.suffix == '.parquet' else pd.read_csv(file, low_memory=False)

audit = []
for file in FILES:
    df = read_raw(file)
    audit.append({'dataset': file.name, 'rows': len(df), 'columns': len(df.columns),
                  'duplicate_rows': int(df.duplicated().sum()),
                  'missing_cells': int(df.isna().sum().sum()),
                  'missing_pct': round(float(df.isna().mean().mean()), 5)})

audit_df = pd.DataFrame(audit).sort_values('dataset')
audit_df

## 2. Jalankan cleaning pipeline aktual

`clean_all()` menjalankan seluruh aturan yang sama dengan pipeline offline dan menulis Parquet ke `data/processed/clean`. Validasi schema, critical null, uniqueness, tipe numerik, dan rentang dilakukan sebelum output ditulis.

In [ ]:
clean = clean_all()
{name: df.shape for name, df in clean.items()}

In [ ]:
quality = validate_cleaned(clean)
display(quality[quality['status'].isin(['FAIL', 'WARN'])].head(50))
print('Total checks:', len(quality))
print('FAIL:', int((quality['status'] == 'FAIL').sum()))
print('WARN:', int((quality['status'] == 'WARN').sum()))

## 3. Ringkasan aturan cleaning

### Usia
`usia_bulan1` dan `usia` dikonversi menjadi numerik, tidak diimputasi, dan wajib berada pada rentang 15–100. Jika missing atau invalid, quality check gagal.

### Demografi
`provinsi`, `jenis_kelamin`, dan `pekerjaan` distandardisasi ke kategori canonical. `kisaran_pendapatan` dan `channel` tidak ditebak; missing tetap missing karena nilainya tidak dapat dipulihkan secara valid.

### Panel produk dan segmen
Status produk menggunakan `ffill()` lalu sisa missing menjadi `0`. `segmen` menggunakan `ffill()` dan `bfill()` dalam nasabah yang sama.

### Transaksi dan saldo
`jumlah_transaksi` diisi median per pasangan `from_id`-`to_id`, dengan fallback `1`. Nominal outlier positif lebih dari 100 kali median kelompok dibagi 1.000. `saldo_akhir_bulan_rp` yang missing diisi dengan `saldo_rata2_rp * median_ratio` dan diberi flag imputasi.

In [ ]:
n = clean['nasabah']
m = clean['nasabah_bulanan']
s = clean['saldo']

age_check = pd.DataFrame({
    'dataset': ['nasabah', 'nasabah_bulanan'],
    'missing_age': [int(n['usia_bulan1'].isna().sum()), int(m['usia'].isna().sum())],
    'invalid_age': [int((~n['usia_bulan1'].between(15, 100)).sum()), int((~m['usia'].between(15, 100)).sum())]
})
display(age_check)

imputation_summary = pd.DataFrame({
    'flag': ['saldo_akhir_bulan_rp_diimputasi'],
    'count': [int(s['saldo_akhir_bulan_rp_diimputasi'].sum())],
    'remaining_missing': [int(s['saldo_akhir_bulan_rp'].isna().sum())]
})
imputation_summary

In [ ]:
flag_columns = [c for df in clean.values() for c in df.columns if c.endswith('_imputed') or c.endswith('_corrected')]
flag_columns = sorted(set(flag_columns))
flag_summary = []
for name, df in clean.items():
    for col in flag_columns:
        if col in df:
            flag_summary.append({'dataset': name, 'flag': col, 'count': int(df[col].fillna(0).sum())})
pd.DataFrame(flag_summary)

## 4. Evaluasi terhadap truth sintetis

Evaluasi ini hanya untuk mengukur generator sintetis. File truth tidak dipakai untuk memperbaiki data.

In [ ]:
evaluation = evaluate_cleaning(clean)
evaluation

## 5. Lokasi output

Output pipeline tersimpan di `data/processed/clean/`, termasuk file Parquet dan `data_quality_report.csv`. Notebook ini sengaja menggunakan fungsi dari `clean.py` agar aturan dokumentasi dan pipeline tidak terpisah.